In [17]:
!pwd

/Users/hgkahng/Workspaces/soft-prompt/notebooks/sst-2


In [18]:
import os
import sys

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
ROOT_DIR = Path("../../").resolve()
WRITE_DIR = ROOT_DIR / "data/sst/"
print(WRITE_DIR)

/Users/hgkahng/Workspaces/soft-prompt/data/sst


In [20]:
if not WRITE_DIR.exists():
    WRITE_DIR.mkdir()

In [21]:
from datasets import load_dataset

In [22]:
ds = load_dataset("stanfordnlp/sst", trust_remote_code=True)

Using the latest cached version of the module from /Users/hgkahng/.cache/huggingface/modules/datasets_modules/datasets/stanfordnlp--sst/b8a7889ef01c5d3ae8c379b84cc4080f8aad3ac2bc538701cbe0ac6416fb76ff (last modified on Mon May  5 12:30:23 2025) since it couldn't be found locally at stanfordnlp/sst, or remotely on the Hugging Face Hub.


In [23]:
ds

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'tokens', 'tree'],
        num_rows: 8544
    })
    validation: Dataset({
        features: ['sentence', 'label', 'tokens', 'tree'],
        num_rows: 1101
    })
    test: Dataset({
        features: ['sentence', 'label', 'tokens', 'tree'],
        num_rows: 2210
    })
})

In [ ]:
df_train = ds['train'].to_pandas()
df_validation = ds['validation'].to_pandas()
df_test = ds['test'].to_pandas()

## Compute Embeddings
- Model: `text-embedding-3-small`

In [26]:
import asyncio

from langchain_openai import OpenAIEmbeddings
from typing import List


async def batch_embed_documents(texts: List[str], batch_size: int = 10):
    
    # Initialize the embeddings class
    embedder = OpenAIEmbeddings(model='text-embedding-3-small',
                                max_retries=5,
                                request_timeout=120.,)

    # Create batches of texts
    batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
    print(f">> Split {len(texts)} documents into {len(batches)} batches")
    
    embeddings = []
    
    # Process each batch
    for i, batch in enumerate(batches):
        print(f">> Processing batch {i+1:>4,}/{len(batches):>4,} ({len(batch)} documents)")
        
        # Get embeddings for the current batch
        batch_embeddings = await embedder.aembed_documents(batch)
        embeddings.extend(batch_embeddings)
        
        print(f">>  Completed batch {i+1:>4,}/{len(batches):>4,}")
        
        # Optional: Add a delay between batches to avoid rate limits
        if i < len(batches) - 1:
            await asyncio.sleep(0.5)  # 500ms delay between batches
    
    print(f">> Generated {len(embeddings):,} embeddings in total")

    return np.array(embeddings)

Training Data

In [28]:
X_train = await batch_embed_documents(df_train['sentence'], batch_size=100)

>> Split 8544 documents into 86 batches
>> Processing batch    1/  86 (100 documents)
>>  Completed batch    1/  86
>> Processing batch    2/  86 (100 documents)
>>  Completed batch    2/  86
>> Processing batch    3/  86 (100 documents)
>>  Completed batch    3/  86
>> Processing batch    4/  86 (100 documents)
>>  Completed batch    4/  86
>> Processing batch    5/  86 (100 documents)
>>  Completed batch    5/  86
>> Processing batch    6/  86 (100 documents)
>>  Completed batch    6/  86
>> Processing batch    7/  86 (100 documents)
>>  Completed batch    7/  86
>> Processing batch    8/  86 (100 documents)
>>  Completed batch    8/  86
>> Processing batch    9/  86 (100 documents)
>>  Completed batch    9/  86
>> Processing batch   10/  86 (100 documents)
>>  Completed batch   10/  86
>> Processing batch   11/  86 (100 documents)
>>  Completed batch   11/  86
>> Processing batch   12/  86 (100 documents)
>>  Completed batch   12/  86
>> Processing batch   13/  86 (100 documents)
>>

In [31]:
print(X_train.shape)

(8544, 1536)


In [30]:
MODEL = "text-embedding-3-small"
emb_save_dir = WRITE_DIR / f"embeddings/openai/{MODEL}"
os.makedirs(emb_save_dir, exist_ok=True)
print("Embeddings will be saved to:", emb_save_dir)

Embeddings will be saved to: /Users/hgkahng/Workspaces/soft-prompt/data/sst/embeddings/openai/text-embedding-3-small


In [32]:
np.save(emb_save_dir / "train.features.npy", X_train)

In [33]:
y_train = df_train['label'].values
print(y_train.shape)

(8544,)


In [34]:
np.save(emb_save_dir / "train.labels.npy", y_train)

Validation Data

In [35]:
X_val = await batch_embed_documents(df_validation['sentence'], batch_size=100)

>> Split 1101 documents into 12 batches
>> Processing batch    1/  12 (100 documents)
>>  Completed batch    1/  12
>> Processing batch    2/  12 (100 documents)
>>  Completed batch    2/  12
>> Processing batch    3/  12 (100 documents)
>>  Completed batch    3/  12
>> Processing batch    4/  12 (100 documents)
>>  Completed batch    4/  12
>> Processing batch    5/  12 (100 documents)
>>  Completed batch    5/  12
>> Processing batch    6/  12 (100 documents)
>>  Completed batch    6/  12
>> Processing batch    7/  12 (100 documents)
>>  Completed batch    7/  12
>> Processing batch    8/  12 (100 documents)
>>  Completed batch    8/  12
>> Processing batch    9/  12 (100 documents)
>>  Completed batch    9/  12
>> Processing batch   10/  12 (100 documents)
>>  Completed batch   10/  12
>> Processing batch   11/  12 (100 documents)
>>  Completed batch   11/  12
>> Processing batch   12/  12 (1 documents)
>>  Completed batch   12/  12
>> Generated 1,101 embeddings in total


In [36]:
print(X_val.shape)

(1101, 1536)


In [37]:
np.save(emb_save_dir / "validation.features.npy", X_val)

In [38]:
y_val = df_validation['label'].values
print(y_val.shape)

(1101,)


In [39]:
np.save(emb_save_dir / "validation.labels.npy", y_val)

Test Data

In [40]:
X_test = await batch_embed_documents(df_test['sentence'], batch_size=100)

>> Split 2210 documents into 23 batches
>> Processing batch    1/  23 (100 documents)
>>  Completed batch    1/  23
>> Processing batch    2/  23 (100 documents)
>>  Completed batch    2/  23
>> Processing batch    3/  23 (100 documents)
>>  Completed batch    3/  23
>> Processing batch    4/  23 (100 documents)
>>  Completed batch    4/  23
>> Processing batch    5/  23 (100 documents)
>>  Completed batch    5/  23
>> Processing batch    6/  23 (100 documents)
>>  Completed batch    6/  23
>> Processing batch    7/  23 (100 documents)
>>  Completed batch    7/  23
>> Processing batch    8/  23 (100 documents)
>>  Completed batch    8/  23
>> Processing batch    9/  23 (100 documents)
>>  Completed batch    9/  23
>> Processing batch   10/  23 (100 documents)
>>  Completed batch   10/  23
>> Processing batch   11/  23 (100 documents)
>>  Completed batch   11/  23
>> Processing batch   12/  23 (100 documents)
>>  Completed batch   12/  23
>> Processing batch   13/  23 (100 documents)
>>

In [41]:
print(X_test.shape)

(2210, 1536)


In [42]:
np.save(emb_save_dir / "test.features.npy", X_test)

In [43]:
y_test = df_test['label'].values
print(y_test.shape)

(2210,)


In [44]:
np.save(emb_save_dir / "test.labels.npy", y_test)